# Week 1 - M&A Prediction Foundation Notebook

本 notebook 根据 `code/README.md` 的 Week 1 目标完成项目基础搭建：范围定义、KPI、目标 universe、数据接入 smoke tests、目录结构、实验记录和第一周交付清单。

Week 1 deliverables:
- Scoping doc: project objective, prediction horizon, KPIs, success criteria
- Data-access confirmation: SEC EDGAR, market data, options data, fund disclosures
- Infrastructure checklist: local folders, config, run manifest, experiment tracking placeholder
- Initial target universe seed and historical training-window definition
- Literature review tracker for M&A prediction, merger arbitrage, and informed trading detection

## 1. Environment and Project Paths

Run this notebook from either the repository root or the `code/` folder. Generated artifacts are written under `data/`, `outputs/week1/`, and `reports/week1/`.

In [1]:
from __future__ import annotations

import csv
import datetime as dt
import importlib.util
import json
import os
import platform
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "README.md").exists() and (candidate / "code" / "README.md").exists():
            return candidate
    return cwd


PROJECT_ROOT = resolve_project_root()
CODE_DIR = PROJECT_ROOT / "code"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "week1"
REPORT_DIR = PROJECT_ROOT / "reports" / "week1"
LOG_DIR = PROJECT_ROOT / "logs"

for path in [
    RAW_DIR / "sec",
    RAW_DIR / "market",
    RAW_DIR / "options",
    RAW_DIR / "funds",
    INTERIM_DIR,
    PROCESSED_DIR,
    OUTPUT_DIR,
    REPORT_DIR,
    LOG_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

Project root: /Users/cindyfu/Desktop/UCB/Industry Project - JPM
Python: 3.11.15
Platform: macOS-26.4-arm64-arm-64bit


## 2. Scope, KPIs, and Success Criteria

第一周先把预测任务收紧成可交付版本：以公司为单位，预测未来 3-6 个月是否发生 M&A 相关事件。后续可以拆分 target、acquirer 和 pair-level compatibility。

In [2]:
PROJECT_CONFIG = {
    "project_name": "mna_prediction_market_signals_genai",
    "week": 1,
    "as_of_date": dt.date.today().isoformat(),
    "prediction_unit": "company-quarter / company-month",
    "prediction_horizon_months": [3, 6],
    "primary_label": "acquired_or_announced_target_within_6_months",
    "secondary_labels": [
        "announced_acquirer_within_6_months",
        "strategically_compatible_pair_within_6_months",
    ],
    "initial_universe": "S&P 500 seed list for prototype; Russell 1000 target for full build",
    "training_window": {
        "prototype_start": "2018-01-01",
        "prototype_end": "2024-12-31",
        "live_scoring_start": "2025-01-01",
    },
    "ranking_metrics": ["precision_at_50", "hit_rate_at_k", "roc_auc", "event_capture_rate", "brier_score"],
    "point_in_time_rule": "All features must be timestamped and available before the prediction date.",
}

success_criteria = pd.DataFrame(
    [
        {"metric": "Precision@50", "definition": "Actual acquisition targets in top 50 ranked names", "target": "10-15 hits"},
        {"metric": "Hit Rate@K", "definition": "Share of top-K names that become M&A targets", "target": "> 20%"},
        {"metric": "ROC-AUC", "definition": "Discrimination across full company universe", "target": "> 0.70"},
        {"metric": "Event Capture Rate", "definition": "Share of actual events appearing in top-ranked tier", "target": "> 30%"},
        {"metric": "Brier Score", "definition": "Probability calibration for deal completion / event likelihood", "target": "< 0.15"},
    ]
)

config_path = OUTPUT_DIR / "project_config.json"
config_path.write_text(json.dumps(PROJECT_CONFIG, indent=2), encoding="utf-8")
success_criteria.to_csv(REPORT_DIR / "success_criteria.csv", index=False)

success_criteria

,metric,definition,target
0,Precision@50,Actual acquisition targets in top 50 ranked names,10-15 hits
1,Hit Rate@K,Share of top-K names that become M&A targets,> 20%
2,ROC-AUC,Discrimination across full company universe,> 0.70
3,Event Capture Rate,Share of actual events appearing in top-ranked...,> 30%
4,Brier Score,Probability calibration for deal completion / ...,< 0.15


## 3. Dependency Audit

The notebook uses the standard library and `pandas` by default. Optional packages are checked but not required for the Week 1 scaffold.

In [3]:
required_packages = ["pandas"]
optional_packages = ["numpy", "requests", "yfinance", "sklearn", "mlflow", "wandb", "sec_edgar_downloader"]


def package_status(package: str, required: bool) -> dict:
    spec = importlib.util.find_spec(package)
    return {
        "package": package,
        "required": required,
        "installed": spec is not None,
        "purpose": {
            "pandas": "DataFrames and CSV outputs",
            "numpy": "Numerical operations",
            "requests": "Convenient HTTP client; stdlib urllib is used as fallback",
            "yfinance": "Prototype market data pull",
            "sklearn": "Baseline models and metrics from Week 5 onward",
            "mlflow": "Local experiment tracking",
            "wandb": "Cloud experiment tracking option",
            "sec_edgar_downloader": "Optional SEC filings downloader wrapper",
        }.get(package, ""),
    }


dependency_audit = pd.DataFrame(
    [package_status(pkg, True) for pkg in required_packages]
    + [package_status(pkg, False) for pkg in optional_packages]
)
dependency_audit.to_csv(REPORT_DIR / "dependency_audit.csv", index=False)
dependency_audit

,package,required,installed,purpose
0,pandas,True,True,DataFrames and CSV outputs
1,numpy,False,True,Numerical operations
2,requests,False,False,Convenient HTTP client; stdlib urllib is used ...
3,yfinance,False,False,Prototype market data pull
4,sklearn,False,True,Baseline models and metrics from Week 5 onward
5,mlflow,False,False,Local experiment tracking
6,wandb,False,False,Cloud experiment tracking option
7,sec_edgar_downloader,False,False,Optional SEC filings downloader wrapper


## 4. Data-Access Inventory

Week 1 的核心不是把所有数据拉满，而是确认每类数据的 owner、访问方式、替代方案和阻塞点。

In [4]:
data_access_inventory = pd.DataFrame(
    [
        {
            "source": "SEC EDGAR company tickers and filings",
            "category": "Corporate disclosures",
            "access_method": "Public SEC JSON / submissions API",
            "week1_status": "test_with_smoke_call",
            "backup": "SEC bulk companyfacts and full-index files",
            "notes": "Requires a descriptive SEC_USER_AGENT environment variable.",
        },
        {
            "source": "Market prices and volumes",
            "category": "Market data",
            "access_method": "Bloomberg / Refinitiv / WRDS preferred; yfinance for prototype",
            "week1_status": "prototype_optional",
            "backup": "Stooq, Nasdaq Data Link, local CSV exports",
            "notes": "Production build should use licensed point-in-time data.",
        },
        {
            "source": "Options chains, OI, IV, skew",
            "category": "Options data",
            "access_method": "OptionMetrics / Bloomberg / Refinitiv / WRDS",
            "week1_status": "access_needed",
            "backup": "Cboe DataShop / provider export / limited yfinance live chains",
            "notes": "Historical options data is usually licensed.",
        },
        {
            "source": "ETF and mutual fund holdings",
            "category": "Fund disclosures",
            "access_method": "ETF issuer daily files, SEC N-PORT/N-CEN, Morningstar/FactSet if available",
            "week1_status": "access_needed",
            "backup": "Issuer websites and SEC filings parser",
            "notes": "Normalize identifiers before fund-skill scoring.",
        },
        {
            "source": "M&A ground truth",
            "category": "Labels",
            "access_method": "SDC / Capital IQ / FactSet / Refinitiv preferred",
            "week1_status": "access_needed",
            "backup": "SEC 8-K deal announcements, press releases, Wikipedia/SEC sample for prototype",
            "notes": "Need announcement date, target, acquirer, status, close date, deal value.",
        },
    ]
)

data_access_inventory.to_csv(REPORT_DIR / "data_access_inventory.csv", index=False)
data_access_inventory

,source,category,access_method,week1_status,backup,notes
0,SEC EDGAR company tickers and filings,Corporate disclosures,Public SEC JSON / submissions API,test_with_smoke_call,SEC bulk companyfacts and full-index files,Requires a descriptive SEC_USER_AGENT environm...
1,Market prices and volumes,Market data,Bloomberg / Refinitiv / WRDS preferred; yfinan...,prototype_optional,"Stooq, Nasdaq Data Link, local CSV exports",Production build should use licensed point-in-...
2,"Options chains, OI, IV, skew",Options data,OptionMetrics / Bloomberg / Refinitiv / WRDS,access_needed,Cboe DataShop / provider export / limited yfin...,Historical options data is usually licensed.
3,ETF and mutual fund holdings,Fund disclosures,"ETF issuer daily files, SEC N-PORT/N-CEN, Morn...",access_needed,Issuer websites and SEC filings parser,Normalize identifiers before fund-skill scoring.
4,M&A ground truth,Labels,SDC / Capital IQ / FactSet / Refinitiv preferred,access_needed,"SEC 8-K deal announcements, press releases, Wi...","Need announcement date, target, acquirer, stat..."


## 5. SEC EDGAR Smoke Test

SEC asks automated clients to send a descriptive User-Agent. Before running live calls, set:

```bash
export SEC_USER_AGENT="Your Name your.email@example.com"
```

If it is not set, this cell still records a clear blocked status rather than failing silently.

In [5]:
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "")
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"


def fetch_json(url: str, headers: dict[str, str] | None = None, timeout: int = 20) -> dict:
    request = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def sec_headers() -> dict[str, str]:
    if not SEC_USER_AGENT:
        raise ValueError("SEC_USER_AGENT is not set. Use: export SEC_USER_AGENT='Name email@example.com'")
    return {"User-Agent": SEC_USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "www.sec.gov"}


def run_sec_company_tickers_smoke_test() -> tuple[pd.DataFrame, dict]:
    started = time.time()
    status = {
        "source": "SEC company_tickers.json",
        "url": SEC_TICKERS_URL,
        "checked_at": dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat(),
        "status": "not_run",
        "row_count": 0,
        "elapsed_seconds": None,
        "error": "",
    }
    try:
        raw = fetch_json(SEC_TICKERS_URL, headers=sec_headers())
        rows = list(raw.values())
        frame = pd.DataFrame(rows)
        frame["cik_str"] = frame["cik_str"].astype(str).str.zfill(10)
        frame = frame.rename(columns={"cik_str": "cik", "ticker": "ticker", "title": "company_name"})
        frame = frame[["ticker", "cik", "company_name"]].sort_values("ticker").reset_index(drop=True)
        frame.to_csv(RAW_DIR / "sec" / "company_tickers.csv", index=False)
        status.update({"status": "ok", "row_count": len(frame)})
    except Exception as exc:
        frame = pd.DataFrame(columns=["ticker", "cik", "company_name"])
        status.update({"status": "blocked", "error": repr(exc)})
    finally:
        status["elapsed_seconds"] = round(time.time() - started, 2)
    return frame, status


sec_tickers, sec_status = run_sec_company_tickers_smoke_test()
pd.DataFrame([sec_status]).to_csv(REPORT_DIR / "sec_smoke_test.csv", index=False)
print(sec_status)
sec_tickers.head()

{'source': 'SEC company_tickers.json', 'url': 'https://www.sec.gov/files/company_tickers.json', 'checked_at': '2026-06-10T22:56:12+00:00', 'status': 'blocked', 'row_count': 0, 'elapsed_seconds': 0.0, 'error': 'ValueError("SEC_USER_AGENT is not set. Use: export SEC_USER_AGENT=\'Name email@example.com\'")'}


,ticker,cik,company_name


## 6. Initial Universe Seed

Prototype seed list uses large-cap companies across financials, technology, healthcare, energy, consumer, industrials, and communications. In production, replace this with a point-in-time S&P 500 or Russell 1000 membership table.

In [ ]:
prototype_universe = pd.DataFrame(
    [
        {"ticker": "JPM", "sector": "Financials", "role_hint": "acquirer_candidate"},
        {"ticker": "BAC", "sector": "Financials", "role_hint": "acquirer_candidate"},
        {"ticker": "MSFT", "sector": "Information Technology", "role_hint": "acquirer_candidate"},
        {"ticker": "AAPL", "sector": "Information Technology", "role_hint": "acquirer_candidate"},
        {"ticker": "NVDA", "sector": "Information Technology", "role_hint": "acquirer_candidate"},
        {"ticker": "ADBE", "sector": "Information Technology", "role_hint": "target_or_acquirer"},
        {"ticker": "CRM", "sector": "Information Technology", "role_hint": "target_or_acquirer"},
        {"ticker": "PFE", "sector": "Health Care", "role_hint": "acquirer_candidate"},
        {"ticker": "MRK", "sector": "Health Care", "role_hint": "acquirer_candidate"},
        {"ticker": "AMGN", "sector": "Health Care", "role_hint": "target_or_acquirer"},
        {"ticker": "XOM", "sector": "Energy", "role_hint": "acquirer_candidate"},
        {"ticker": "CVX", "sector": "Energy", "role_hint": "acquirer_candidate"},
        {"ticker": "DIS", "sector": "Communication Services", "role_hint": "target_or_acquirer"},
        {"ticker": "CMCSA", "sector": "Communication Services", "role_hint": "acquirer_candidate"},
        {"ticker": "WMT", "sector": "Consumer Staples", "role_hint": "acquirer_candidate"},
        {"ticker": "TGT", "sector": "Consumer Staples", "role_hint": "target_or_acquirer"},
        {"ticker": "BA", "sector": "Industrials", "role_hint": "target_or_acquirer"},
        {"ticker": "GE", "sector": "Industrials", "role_hint": "portfolio_optimization_watch"},
    ]
)

if not sec_tickers.empty:
    prototype_universe = prototype_universe.merge(sec_tickers, how="left", on="ticker")
else:
    prototype_universe["cik"] = None
    prototype_universe["company_name"] = None

prototype_universe["universe_version"] = "week1_prototype_seed"
prototype_universe.to_csv(INTERIM_DIR / "universe_seed_week1.csv", index=False)
prototype_universe

## 7. Market Data Smoke Test

This prototype uses `yfinance` only if installed. Licensed point-in-time data should replace this for research-grade modeling.

In [ ]:
def run_yfinance_smoke_test(tickers: list[str], start: str, end: str) -> tuple[pd.DataFrame, dict]:
    started = time.time()
    status = {
        "source": "yfinance",
        "tickers": ",".join(tickers),
        "start": start,
        "end": end,
        "checked_at": dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat(),
        "status": "not_run",
        "row_count": 0,
        "elapsed_seconds": None,
        "error": "",
    }
    if importlib.util.find_spec("yfinance") is None:
        status.update({"status": "blocked", "error": "Optional package yfinance is not installed.", "elapsed_seconds": round(time.time() - started, 2)})
        return pd.DataFrame(), status

    try:
        import yfinance as yf

        prices = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False, group_by="ticker")
        if prices.empty:
            raise ValueError("No rows returned from yfinance.")
        out_path = RAW_DIR / "market" / "yfinance_smoke_prices.csv"
        prices.to_csv(out_path)
        status.update({"status": "ok", "row_count": int(len(prices))})
        return prices, status
    except Exception as exc:
        status.update({"status": "blocked", "error": repr(exc)})
        return pd.DataFrame(), status
    finally:
        status["elapsed_seconds"] = round(time.time() - started, 2)


market_prices, market_status = run_yfinance_smoke_test(["JPM", "MSFT", "PFE"], "2024-01-01", "2024-02-01")
pd.DataFrame([market_status]).to_csv(REPORT_DIR / "market_smoke_test.csv", index=False)
print(market_status)
market_prices.head()

## 8. Feature Store Schema Draft

Before feature engineering starts, define the common keys and point-in-time fields so market, NLP, options, and fund signals can merge cleanly later.

In [6]:
feature_store_schema = pd.DataFrame(
    [
        {"column": "as_of_date", "type": "date", "required": True, "description": "Prediction date; feature availability cutoff"},
        {"column": "ticker", "type": "string", "required": True, "description": "Trading ticker as of as_of_date"},
        {"column": "cik", "type": "string", "required": False, "description": "SEC Central Index Key, zero padded"},
        {"column": "company_name", "type": "string", "required": False, "description": "Issuer name"},
        {"column": "sector", "type": "string", "required": False, "description": "Sector classification"},
        {"column": "market_abnormal_return_20d", "type": "float", "required": False, "description": "CAPM/sector-adjusted 20-day return"},
        {"column": "market_volume_zscore_20d", "type": "float", "required": False, "description": "20-day abnormal volume z-score"},
        {"column": "options_call_oi_anomaly", "type": "float", "required": False, "description": "Call open-interest anomaly score"},
        {"column": "options_put_call_divergence", "type": "float", "required": False, "description": "Put/call ratio divergence from sector norms"},
        {"column": "nlp_strategic_alternatives_score", "type": "float", "required": False, "description": "LLM-extracted strategic alternatives signal"},
        {"column": "nlp_portfolio_optimization_score", "type": "float", "required": False, "description": "LLM-extracted divestiture/restructuring signal"},
        {"column": "fund_rebalance_signal", "type": "float", "required": False, "description": "Signal from high-skill merger-arb fund holdings changes"},
        {"column": "label_target_within_6m", "type": "int", "required": False, "description": "1 if company is announced as acquisition target within 6 months"},
        {"column": "label_acquirer_within_6m", "type": "int", "required": False, "description": "1 if company is announced as acquirer within 6 months"},
    ]
)

feature_store_schema.to_csv(REPORT_DIR / "feature_store_schema_draft.csv", index=False)
feature_store_schema

,column,type,required,description
0,as_of_date,date,True,Prediction date; feature availability cutoff
1,ticker,string,True,Trading ticker as of as_of_date
2,cik,string,False,"SEC Central Index Key, zero padded"
3,company_name,string,False,Issuer name
4,sector,string,False,Sector classification
5,market_abnormal_return_20d,float,False,CAPM/sector-adjusted 20-day return
6,market_volume_zscore_20d,float,False,20-day abnormal volume z-score
7,options_call_oi_anomaly,float,False,Call open-interest anomaly score
8,options_put_call_divergence,float,False,Put/call ratio divergence from sector norms
9,nlp_strategic_alternatives_score,float,False,LLM-extracted strategic alternatives signal


## 9. Literature Review Tracker

Use this as a structured backlog for Week 1 research. Add papers/articles and summarize whether each source informs labels, features, evaluation, or trading simulation.

In [ ]:
literature_tracker = pd.DataFrame(
    [
        {
            "topic": "M&A target prediction",
            "search_query": "machine learning prediction acquisition targets public firms features",
            "why_it_matters": "Baseline labels, rare-event modeling, target-specific predictors",
            "status": "to_review",
        },
        {
            "topic": "Merger arbitrage completion risk",
            "search_query": "merger arbitrage deal completion probability determinants spread",
            "why_it_matters": "Fund scoring and announced-deal probability calibration",
            "status": "to_review",
        },
        {
            "topic": "Informed options trading before M&A",
            "search_query": "options trading volume before merger acquisition announcements informed trading",
            "why_it_matters": "Options anomaly features and event-window design",
            "status": "to_review",
        },
        {
            "topic": "SEC filing language and strategic intent",
            "search_query": "SEC filings strategic alternatives textual analysis mergers acquisitions prediction",
            "why_it_matters": "Prompt design and NLP signal validation",
            "status": "to_review",
        },
        {
            "topic": "Look-ahead and survivorship bias",
            "search_query": "point in time universe survivorship bias event study machine learning finance",
            "why_it_matters": "Research design and backtest credibility",
            "status": "to_review",
        },
    ]
)

literature_tracker.to_csv(REPORT_DIR / "literature_review_tracker.csv", index=False)
literature_tracker

## 10. Week 1 Checklist and Run Manifest

This section writes the final Week 1 status report artifacts. `blocked` means the item needs credentials, paid data, or a package install before it can be fully completed.

In [ ]:
week1_checklist = pd.DataFrame(
    [
        {"workstream": "Scope", "item": "Objective, prediction horizon, and labels defined", "status": "complete", "artifact": str(config_path.relative_to(PROJECT_ROOT))},
        {"workstream": "KPI", "item": "Ranking metrics and success criteria defined", "status": "complete", "artifact": "reports/week1/success_criteria.csv"},
        {"workstream": "Infrastructure", "item": "Local data/output/report/log folders created", "status": "complete", "artifact": "data/, outputs/week1/, reports/week1/, logs/"},
        {"workstream": "Dependencies", "item": "Required and optional Python packages audited", "status": "complete", "artifact": "reports/week1/dependency_audit.csv"},
        {"workstream": "SEC", "item": "EDGAR company ticker smoke test", "status": sec_status["status"], "artifact": "reports/week1/sec_smoke_test.csv"},
        {"workstream": "Market", "item": "Market data prototype smoke test", "status": market_status["status"], "artifact": "reports/week1/market_smoke_test.csv"},
        {"workstream": "Options", "item": "Historical options data provider identified", "status": "access_needed", "artifact": "reports/week1/data_access_inventory.csv"},
        {"workstream": "Funds", "item": "Fund holdings source plan documented", "status": "access_needed", "artifact": "reports/week1/data_access_inventory.csv"},
        {"workstream": "Universe", "item": "Prototype target universe seed written", "status": "complete", "artifact": "data/interim/universe_seed_week1.csv"},
        {"workstream": "Research", "item": "Literature review backlog created", "status": "complete", "artifact": "reports/week1/literature_review_tracker.csv"},
    ]
)

run_manifest = {
    "run_id": dt.datetime.now(dt.UTC).strftime("week1_%Y%m%dT%H%M%SZ"),
    "created_at_utc": dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook": "code/week1_foundation.ipynb",
    "config": PROJECT_CONFIG,
    "sec_smoke_test": sec_status,
    "market_smoke_test": market_status,
    "artifacts": sorted(str(path.relative_to(PROJECT_ROOT)) for path in REPORT_DIR.glob("*.csv"))
    + sorted(str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_DIR.glob("*.json")),
}

week1_checklist.to_csv(REPORT_DIR / "week1_checklist.csv", index=False)
(OUTPUT_DIR / "week1_run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")

week1_checklist

## Next Steps for Week 2

1. Replace prototype universe with point-in-time S&P 500 or Russell 1000 membership.
2. Confirm licensed market/options provider and write raw extracts into `data/raw/market/` and `data/raw/options/`.
3. Build feature functions for returns, abnormal returns, rolling volatility, liquidity, volume z-scores, IV rank, put/call divergence, and skew shifts.
4. Add data quality checks for ticker mapping, missing prices, duplicate rows, corporate actions, and feature availability dates.